In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/bvanntruong/housing-sql-project/Nashville Housing.csv


In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
import sqlite3

In [3]:
df=pd.read_csv("/kaggle/input/datasets/bvanntruong/housing-sql-project/Nashville Housing.csv")
df.info()
df.head()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56477 entries, 0 to 56476
Data columns (total 19 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   UniqueID         56477 non-null  int64  
 1   ParcelID         56477 non-null  object 
 2   LandUse          56477 non-null  object 
 3   PropertyAddress  56448 non-null  object 
 4   SaleDate         56477 non-null  object 
 5   SalePrice        56477 non-null  object 
 6   LegalReference   56477 non-null  object 
 7   SoldAsVacant     56477 non-null  object 
 8   OwnerName        25261 non-null  object 
 9   OwnerAddress     26015 non-null  object 
 10  Acreage          26015 non-null  float64
 11  TaxDistrict      26015 non-null  object 
 12  LandValue        26015 non-null  float64
 13  BuildingValue    26015 non-null  float64
 14  TotalValue       26015 non-null  float64
 15  YearBuilt        24163 non-null  float64
 16  Bedrooms         24157 non-null  float64
 17  FullBath    

,UniqueID,Acreage,LandValue,BuildingValue,TotalValue,YearBuilt,Bedrooms,FullBath,HalfBath
count,56477.000000,26015.000000,2.601500e+04,2.601500e+04,2.601500e+04,24163.000000,24157.000000,24275.000000,24144.000000
mean,28334.001133,0.498923,6.906856e+04,1.607847e+05,2.323754e+05,1963.744899,3.089912,1.886014,0.283921
std,16352.590651,1.570454,1.060401e+05,2.067999e+05,2.810643e+05,26.542982,0.852869,0.961515,0.487881
min,0.000000,0.010000,1.000000e+02,0.000000e+00,1.000000e+02,1799.000000,0.000000,0.000000,0.000000
25%,14186.000000,0.180000,2.100000e+04,7.590000e+04,1.028000e+05,1948.000000,3.000000,1.000000,0.000000
50%,28313.000000,0.270000,2.880000e+04,1.114000e+05,1.485000e+05,1960.000000,3.000000,2.000000,0.000000
75%,42513.000000,0.450000,6.000000e+04,1.807000e+05,2.683500e+05,1983.000000,3.000000,2.000000,1.000000
max,56635.000000,160.060000,2.772000e+06,1.297180e+07,1.394040e+07,2017.000000,11.000000,10.000000,3.000000


In [4]:
#missing value check
print(df.isnull().sum())
df["PropertyAddress"]=df.groupby("ParcelID")["PropertyAddress"].transform(lambda x:x.ffill().bfill())
df["OwnerName"]=df["OwnerName"].fillna("Unknown")
df["OwnerAddress"]=df["OwnerAddress"].fillna("Unknown")
df["TaxDistrict"]=df["TaxDistrict"].fillna("Unknown")
x = [
    "Acreage",
    "LandValue",
    "BuildingValue",
    "TotalValue",
    "YearBuilt",
    "Bedrooms",
    "FullBath",
    "HalfBath"
]
for col in x:
    df[col ] =df[col].fillna(df[col].median())
print(df.isnull().sum())

UniqueID               0
ParcelID               0
LandUse                0
PropertyAddress       29
SaleDate               0
SalePrice              0
LegalReference         0
SoldAsVacant           0
OwnerName          31216
OwnerAddress       30462
Acreage            30462
TaxDistrict        30462
LandValue          30462
BuildingValue      30462
TotalValue         30462
YearBuilt          32314
Bedrooms           32320
FullBath           32202
HalfBath           32333
dtype: int64
UniqueID           0
ParcelID           0
LandUse            0
PropertyAddress    0
SaleDate           0
SalePrice          0
LegalReference     0
SoldAsVacant       0
OwnerName          0
OwnerAddress       0
Acreage            0
TaxDistrict        0
LandValue          0
BuildingValue      0
TotalValue         0
YearBuilt          0
Bedrooms           0
FullBath           0
HalfBath           0
dtype: int64


In [5]:
#Duplicate check
print(df.duplicated().sum())

0


In [6]:
#Import SQL
import sqlite3
conn=sqlite3.connect(":memory")
df.to_sql("Housing",conn,index=False,if_exists="replace")

56477

In [7]:
query="""SELECT * FROM Housing WHERE PropertyAddress is NULL"""
result=pd.read_sql_query(query,conn)
result

,UniqueID,ParcelID,LandUse,PropertyAddress,SaleDate,SalePrice,LegalReference,SoldAsVacant,OwnerName,OwnerAddress,Acreage,TaxDistrict,LandValue,BuildingValue,TotalValue,YearBuilt,Bedrooms,FullBath,HalfBath


In [8]:
#Total Sales
query="""select COUNT(*) AS Total_Sales FROM HOUSING"""
result=pd.read_sql_query(query,conn)
result


,Total_Sales
0,56477


In [9]:
#Land Analysis

query="""select LandUse,COUNT(*) AS Total FROM Housing GROUP BY LandUse ORDER BY total DESC
"""
result=pd.read_sql_query(query,conn)
result


,LandUse,Total
0,SINGLE FAMILY,34197
1,RESIDENTIAL CONDO,14080
2,VACANT RESIDENTIAL LAND,3547
3,VACANT RES LAND,1549
4,DUPLEX,1373
5,ZERO LOT LINE,1048
6,CONDO,247
7,RESIDENTIAL COMBO/MISC,95
8,TRIPLEX,92
9,QUADPLEX,39


In [10]:
#price analysis
query="""SELECT 
AVG(SalePrice) AS Average_Price,
MIN(SalePrice) AS Minimum_Price,
MAX(SalePrice) AS Maximum_Price
from Housing;"""
result=pd.read_sql_query(query,conn)
result


,Average_Price,Minimum_Price,Maximum_Price
0,327177.423411,"$1,124,900",999999


In [11]:
#Landuse price
query="""SELECT LandUse,SUM(SalePrice) AS Total_Price FROM Housing Group BY LandUse
ORDER BY Total_Price DESC;"""
result=pd.read_sql_query(query,conn)
result


,LandUse,Total_Price
0,SINGLE FAMILY,9.582738e+09
1,RESIDENTIAL CONDO,6.138406e+09
2,VACANT RESIDENTIAL LAND,1.328873e+09
3,VACANT RES LAND,3.695578e+08
4,DUPLEX,3.566194e+08
5,CONDO,3.112358e+08
6,ZERO LOT LINE,1.301871e+08
7,VACANT COMMERCIAL LAND,5.500000e+07
8,CONDOMINIUM OFC OR OTHER COM CONDO,4.391090e+07
9,RESIDENTIAL COMBO/MISC,3.171303e+07


In [12]:
#LandUse Average price
query="""SELECT LandUse,AVG(SalePrice) AS Average_Price FROM Housing Group BY LandUse
ORDER BY Average_Price DESC;"""
result=pd.read_sql_query(query,conn)
result


,LandUse,Average_Price
0,VACANT COMMERCIAL LAND,3.235294e+06
1,APARTMENT: LOW RISE (BUILT SINCE 1960),2.000000e+06
2,DAY CARE CENTER,1.577500e+06
3,CONDO,1.260064e+06
4,CONDOMINIUM OFC OR OTHER COM CONDO,1.254597e+06
5,PARKING LOT,1.225336e+06
6,LIGHT MANUFACTURING,1.200000e+06
7,FOREST,1.085330e+06
8,CHURCH,8.205735e+05
9,GREENBELT,6.049385e+05


In [13]:
#Time based analysis
query="""SELECT SaleDate FROM Housing LIMIT 10"""
result=pd.read_sql_query(query,conn)
result


,SaleDate
0,"April 9, 2013"
1,"June 10, 2014"
2,"September 26, 2016"
3,"January 29, 2016"
4,"October 10, 2014"
5,"July 16, 2014"
6,"August 28, 2014"
7,"September 27, 2016"
8,"August 14, 2015"
9,"August 29, 2014"


In [14]:
#Year-wise sales
query="""SELECT substr(SaleDate,-4) AS SaleYear,COUNT(*) AS TotalSales FROM Housing GROUP BY substr(SaleDate,-4) ORDER BY SaleYear DESC"""
result=pd.read_sql_query(query,conn)
result


,SaleYear,TotalSales
0,2019,2
1,2016,14071
2,2015,16829
3,2014,14283
4,2013,11292


In [15]:
#year wise average sale
query="""SELECT substr(SaleDate,-4) AS SaleYear,AVG(SalePrice) AS AveragePrice FROM Housing GROUP BY substr(SaleDate,-4) ORDER BY SaleYear DESC"""
result=pd.read_sql_query(query,conn)
result


,SaleYear,AveragePrice
0,2019,118500.000000
1,2016,300982.917206
2,2015,398622.369422
3,2014,334156.720437
4,2013,244549.718473


In [16]:
#Sold as vacant analysis
query="""SELECT SoldAsVacant,COUNT(*) AS Total FROM Housing GROUP BY SOLDAsVACANT"""
result=pd.read_sql_query(query,conn)
result


,SoldAsVacant,Total
0,N,399
1,No,51403
2,Y,52
3,Yes,4623


In [17]:
#Percentage
query="""SELECT SoldAsVacant,COUNT(*) AS Total,
ROUND(COUNT(*) * 100/(SELECT COUNT(*) FROM Housing),2
)AS Percentage FROM housing GROUP BY SOLDAsVacant;"""
result=pd.read_sql_query(query,conn)
result



,SoldAsVacant,Total,Percentage
0,N,399,0.0
1,No,51403,91.0
2,Y,52,0.0
3,Yes,4623,8.0


In [18]:
#Owner analysis
query="""SELECT OwnerName,COUNT(*) AS PropertiesSold FROM Housing WHERE OwnerName is not NULL
GROUP BY OwnerName ORDER BY PropertiesSold DESC LIMIT 10"""
result=pd.read_sql_query(query,conn)
result



,OwnerName,PropertiesSold
0,Unknown,31216
1,"JRG PROPERTIES, LLC",26
2,"HILL 33, LLC",24
3,"SUMMEY, CLARENCE",23
4,"KHAZANOV, MAX",21
5,"FED DEVELOPMENT, LLC",21
6,"WOODLAND STREET PARTNERS, LLC",17
7,"URBANGATE DEVELOPMENT GROUP, LLC",17
8,"KUDZU REAL ESTATE, INC.",17
9,"DEEP RIVER DEVELOPMENT, LLC",16


In [19]:
#Address Analysis
query="""SELECT PropertyAddress,COUNT(*) AS Total FROM Housing GROUP BY PropertyAddress ORDER BY Total DESC LIMIT 10"""
result=pd.read_sql_query(query,conn)
result


,PropertyAddress,Total
0,"1212 LAUREL ST, NASHVILLE",296
1,"600 12TH AVE S, NASHVILLE",208
2,"301 DEMONBREUN ST, NASHVILLE",146
3,"270 TAMPA DR, NASHVILLE",138
4,"1350 ROSA L PARKS BLVD, NASHVILLE",129
5,"415 CHURCH ST, NASHVILLE",118
6,"0 FOREST PARK RD, MADISON",108
7,"1382 RURAL HILL RD, ANTIOCH",89
8,"5025 HILLSBORO PIKE, NASHVILLE",82
9,"6680 CHARLOTTE PIKE, NASHVILLE",80
